# French interview video → English screening evidence

This small prototype processes the supplied 45-second French introduction. It demonstrates **audio decoding → French transcription → Marian French-to-English translation**. It does not classify the applicant, change the production screener, or establish translation accuracy.

The teaching notebook `A0205189B_MarianMT_en_ROMANCE.ipynb` uses English-to-French/ROMANCE models. Here the direction is reversed using `Helsinki-NLP/opus-mt-fr-en`. Marian accepts text; a separate speech recognizer supplies its French input.

Run this notebook in the installed project uv kernel. Heavy inference runs through the package CLI in an optional runtime; this notebook reads its aggregate record. No dependency installation or model definition occurs in notebook cells.

In [1]:
import json
from candidate_screener.config import OUTPUT
record_path = OUTPUT / "multimodal" / "french-interview-demo.json"
record = json.loads(record_path.read_text()) if record_path.exists() else {
    "status": "not_run", "reason": "Run the optional interview_demo CLI first"}
print(json.dumps({k: record.get(k) for k in
    ("status", "duration_seconds", "applicant_review_required", "screening_integration")}, indent=2))

{
  "status": "prototype_complete",
  "duration_seconds": 45.0026875,
  "applicant_review_required": true,
  "screening_integration": "not_enabled"
}


## What was processed

The package decodes only audio to mono 16 kHz PCM. Multilingual Whisper `tiny` runs with explicit French language and `task="transcribe"`. Whisper's English translation task would bypass Marian, so it is not used. The complete clip is transcribed, every timestamped segment is translated with beam width 4. Greedy-versus-beam comparison covers at most the first five segments.

Words are split into chunks against the actual Marian tokenizer budget, including special tokens. The code refuses silent audio, empty transcripts, invalid timestamps, oversized input and unfinished translations. No source text is silently truncated.

In [2]:
asr = record.get("asr", {})
translation = record.get("translation", {})
print(json.dumps({"audio": record.get("audio"), "asr": asr,
    "translation_summary": {k: translation.get(k) for k in
    ("model", "selected_segments", "chunk_count", "beams", "complete", "translated_segments", "full_beam4_seconds", "comparison_greedy_seconds", "token_budget", "full_output_characters", "comparison_scope", "output_characters", "differing_chunks")}}, indent=2))

{
  "audio": {
    "sample_rate": 16000,
    "channels": 1
  },
  "asr": {
    "model": "tiny",
    "language": "fr",
    "task": "transcribe",
    "weights_sha256": "65147644a518d12f04e32d6f3b26facc3f8dd46e5390956a9424a650c0ce22b9",
    "seconds": 4.534655014002055,
    "segment_count": 7
  },
  "translation_summary": {
    "model": "Helsinki-NLP/opus-mt-fr-en",
    "selected_segments": 5,
    "chunk_count": 7,
    "beams": [
      1,
      4
    ],
    "complete": true,
    "translated_segments": 7,
    "full_beam4_seconds": 11.684054514000309,
    "comparison_greedy_seconds": 2.0409802669964847,
    "token_budget": 480,
    "full_output_characters": 689,
    "comparison_scope": "First five ASR segments; full transcript translated with beam 4",
    "output_characters": {
      "1": 554,
      "4": 560
    },
    "differing_chunks": 1
  }
}


## Greedy decoding versus beam search

Beam width 1 is deterministic greedy decoding; width 4 explores multiple candidate sequences. `differing_chunks` counts changed top translations. Different wording is not proof of improved translation. Timings are single-run CPU illustrations, excluding Marian model loading. Model loading and download time are reported separately from inference.

There is no reference transcript or reference translation, so this experiment reports no WER, BLEU or accuracy. The original English video is not treated as gold: it may be a translated or independently scripted recording.

## Privacy and provenance

Full French transcripts, English translations and timestamps stay in ignored `data/processed/multimodal/<media hash>/transcripts.json`. This committed notebook displays aggregate statistics only. Names, addresses, contact information and full applicant text are not exported in notebook outputs. Media, model weights, model revision and dependency versions are recorded below.

## Reviewed translation examples
Only independently reviewed technical excerpts appear here. The redactor is applied again inside this notebook; it does not identify all personal names. A missing preview is reported explicitly, never replaced with invented translations.


In [3]:
from candidate_screener.annotation.redact import redact_text
preview_path = OUTPUT / "multimodal" / "french-interview-preview.json"
if preview_path.exists():
    preview = json.loads(preview_path.read_text())
    rows = [{"start": row["start"], "end": row["end"],
             "french": redact_text(row["french"]),
             "english": redact_text(row["english"]),
             "note": redact_text(row.get("note", ""))} for row in preview["segments"][:5]]
    print(json.dumps(rows, ensure_ascii=False, indent=2)[:2000])
else:
    print("No manually reviewed safe translation preview is available.")


[
  {
    "start": 6.0,
    "end": 12.5,
    "french": "Je suis titulaire d'un bâche-là en informatique qui m'a apporté de solid-bâs en programmation et en analyse de données.",
    "english": "I have a computer tarpaulin that brought me solid-bass in programming and data analysis.",
    "note": "Incoherent education wording: applicant must correct the French transcript and English translation before use."
  },
  {
    "start": 12.5,
    "end": 22.5,
    "french": "Ces dernières années, j'ai développé des applications web et des services back-end avec Python, Java et SQL, en collaborant avec des équipes produits pour fournir des solutions.",
    "english": "In recent years, I have developed web applications and back-end services with Python, Java and SQL, working with product teams to deliver solutions.",
    "note": "Technical stack wording retained in this example; correctness is not independently verified."
  },
  {
    "start": 29.5,
    "end": 35.5,
    "french": "J'aime découvrir

In [4]:
print(json.dumps({"media_sha256": record.get("media_sha256"),
    "marian_revision": translation.get("revision"),
    "marian_artifacts": translation.get("artifact_sha256"),
    "versions": record.get("versions")}, indent=2)[:2000])

{
  "media_sha256": "5896fa7a0093e90d4ac242bf1ecfeb24fc829fe5258e3deaaabb37c44b3c351f",
  "marian_revision": "c4aed37b318c763fd177aa449b44e3b783cc6c02",
  "marian_artifacts": {
    "model.safetensors": "6e3837f34b903802c3d0d670362b997cee6e87584a1108eb3fa89e4625e4424a",
    "target.spm": "173e9f493a668fe396d599e28d414a201193094e6ffd7a4678e5aab0f6d3d838",
    "tokenizer_config.json": "47de9ce87378593016432f8dc657202c03913ab3ce0c15d7f78d51edfc3ff9a3",
    "config.json": "b3be13d046d9899d7aab8cf4ed624d9a79f5776038ba793f6b4d2ce3e02192f7",
    "generation_config.json": "4956fb9a7caaad7579cf8bb789c1e578b8a1cf48a0a8b779fda2f95dd10bbaa5",
    "source.spm": "78d0e717c77053f1c4b856d8661d9cb87c64f083a35418c087b9146300e4f585",
    "vocab.json": "945c604346ce15ce4aff9001001e7f925e336d942c4087017f191871162cbdc4"
  },
  "versions": {
    "torch": "2.2.2",
    "transformers": "4.48.3",
    "openai-whisper": "20250625",
    "imageio-ffmpeg": "0.6.0",
    "sentencepiece": "0.2.2"
  }
}


In [5]:
entity = record.get("entity_audit", {})
print(json.dumps({k: entity.get(k) for k in ("status", "extractor", "counts", "seconds", "complete_field_coverage", "input", "screening_ready", "interpretation")}, indent=2))
print(json.dumps(record.get("duration_audit", {}), indent=2))


{
  "status": "complete",
  "extractor": "GLiNER",
  "counts": {
    "DEGREE": 0,
    "EXPERIENCE_YEARS": 1,
    "KNOWLEDGE": 3,
    "SKILL": 0,
    "TITLE": 0
  },
  "seconds": 2.7435934269960853,
  "complete_field_coverage": true,
  "input": "Unreviewed beam-4 English translation; source contains ASR/MT errors",
  "screening_ready": false,
  "interpretation": "Predicted mentions, not verified qualifications or fit"
}
{
  "source_numeric_duration_mentions": 1,
  "translated_numeric_duration_mentions": 1,
  "same_numeric_duration_multiset": true,
  "human_review_required": true,
  "limitations": "Numeric year expressions only; not total experience or semantic equivalence"
}


## Observed errors and extraction boundary
The actual run contains an incoherent education translation, “computer tarpaulin”, and an occupation rendering, “genius in logic development”. These demonstrate that ASR errors and mistranslations can propagate into extracted evidence. They must be reviewed against the recording; this notebook does not claim the intended qualification or title.
GLiNER counts are predictions on unreviewed English text, not verified applicant facts. A missing DEGREE or TITLE prediction does not mean the applicant lacks either. Numeric-year preservation checks only matching digit expressions; it does not establish total experience or semantic correctness. The manually reviewed preview is approved for display only and does not confirm applicant submission.


## Integration into the intelligent screener

Proposed chat flow: applicant attaches an audio/video file → sees processing status → reviews/corrects French transcript and English translation → confirms submission → reviewed English text becomes CV evidence for existing extraction and matching.

Preserve source timestamps alongside translated sentences so extracted skills, qualifications, titles and experience claims can be traced to the recording. Entity extraction operates on reviewed translated text; the translation is not a new training label. Applicant-review confirmation must precede scoring. Accent, voice characteristics, appearance and gender are not matching features.

The existing ranked matching system remains the baseline. This prototype does not justify replacing it or making Good Fit/Potential Fit/No Fit predictions. Production upload, confirmation and pipeline wiring require the separately approved implementation plan.

References: [MarianMT](https://huggingface.co/docs/transformers/model_doc/marian), [French-to-English model](https://huggingface.co/Helsinki-NLP/opus-mt-fr-en), [Whisper](https://github.com/openai/whisper).